# **Face Shape Classification & Hairstyle Recommendation**
### **Training Pipeline: HOG + SVM & Random Forest (Roboflow Dataset)**
Notebook ini dirancang khusus untuk melatih model klasifikasi bentuk wajah menggunakan dataset dari Roboflow (`faceShape.v4i.folder.zip`).
Hasil model diekspor sebagai `face_shape_svm.pkl` yang kompatibel 100% dengan backend FastAPI proyek PBL.

## **1. Hubungkan ke Google Drive & Ekstrak Dataset**
Upload file `faceShape.v4i.folder.zip` (dari `C:\Users\azure\Downloads\FaceShapeDataset\`) ke Google Drive kamu.

In [ ]:
from google.colab import drive
import zipfile
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path zip dataset di Google Drive (sesuaikan lokasinya jika ditaruh di dalam subfolder)
ZIP_PATH = '/content/drive/MyDrive/faceShape.v4i.folder.zip'
EXTRACT_DIR = '/content/dataset'

if not os.path.exists(ZIP_PATH):
    if os.path.exists('/content/faceShape.v4i.folder.zip'):
        ZIP_PATH = '/content/faceShape.v4i.folder.zip'
    else:
        print(f'File {ZIP_PATH} belum ditemukan! Pastikan sudah mengupload faceShape.v4i.folder.zip')

if os.path.exists(ZIP_PATH):
    print(f'Mengekstrak {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print('Dataset berhasil diekstrak!')
    print('Isi direktori:', os.listdir(EXTRACT_DIR))


## **2. Ekstraksi Fitur HOG (Histogram of Oriented Gradients)**
Spesifikasi ekstraksi disamakan dengan backend:
- Resize: 128x128 (Grayscale)
- Normalisasi intensitas piksel 0..1
- Orientasi: 9, Pixels per cell: (8,8), Cells per block: (2,2)
- Block norm: L2-Hys (Vektor: 8.100 dimensi)

In [ ]:
import cv2
import numpy as np
from PIL import Image
from skimage.feature import hog
from tqdm import tqdm

# Opsional: Inisialisasi MediaPipe Face Detection jika ingin mendeteksi wajah otomatis
try:
    import mediapipe as mp
    _mp_face = mp.solutions.face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5)
    print('MediaPipe Face Detection aktif!')
except Exception:
    _mp_face = None
    print('MediaPipe tidak aktif (menggunakan direct resize).')

# Mapping folder dataset Roboflow ke label canonical proyek
# Roboflow: heart, long, oval, round, square -> Proyek: heart, oblong, oval, round, square
FOLDER_TO_CLASS = {
    'heart': 'heart',
    'long': 'oblong',
    'oval': 'oval',
    'round': 'round',
    'square': 'square'
}

def extract_hog_features(image_path):
    try:
        img = Image.open(image_path).convert('RGB')
        # Deteksi wajah dengan MediaPipe jika aktif
        if _mp_face is not None:
            w, h = img.size
            res = _mp_face.process(np.array(img))
            if res and res.detections:
                best = max(res.detections, key=lambda d: d.location_data.relative_bounding_box.width * d.location_data.relative_bounding_box.height)
                box = best.location_data.relative_bounding_box
                bx, by, bw, bh = int(box.xmin * w), int(box.ymin * h), int(box.width * w), int(box.height * h)
                pad_x, pad_y = int(bw * 0.25), int(bh * 0.25)
                x1, y1 = max(0, bx - pad_x), max(0, by - pad_y)
                x2, y2 = min(w, bx + bw + pad_x), min(h, by + bh + pad_y)
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                sq = max(x2 - x1, y2 - y1)
                img = img.crop((max(0, cx - sq // 2), max(0, cy - sq // 2), min(w, cx + sq // 2), min(h, cy + sq // 2)))
        # Grayscale & resize ke 128x128
        img_gray = img.convert('L').resize((128, 128), Image.Resampling.LANCZOS)
        img_array = np.array(img_gray, dtype=np.float32) / 255.0
        
        features = hog(
            img_array,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            transform_sqrt=True,
            feature_vector=True
        )
        return features
    except Exception as e:
        return None

# Direktori data training
TRAIN_DIR = os.path.join(EXTRACT_DIR, 'train')

# Opsi subset per kelas jika ingin ekstraksi lebih cepat (misal max 1000 gambar/kelas)
# Ubah MAX_SAMPLES_PER_CLASS = None jika ingin memakai SELURUH data (~14.000 gambar)
MAX_SAMPLES_PER_CLASS = 1000

X = []
y = []

print('Mengekstrak fitur HOG dari dataset train...')
for folder_name, canonical_class in FOLDER_TO_CLASS.items():
    class_dir = os.path.join(TRAIN_DIR, folder_name)
    if not os.path.exists(class_dir):
        print(f'Folder {class_dir} tidak ditemukan, skip.')
        continue
        
    all_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    selected_files = all_files[:MAX_SAMPLES_PER_CLASS] if MAX_SAMPLES_PER_CLASS else all_files
    print(f'Memproses kelas {canonical_class} (dari folder {folder_name}): {len(selected_files)} gambar...')
    
    for fname in tqdm(selected_files):
        fpath = os.path.join(class_dir, fname)
        feat = extract_hog_features(fpath)
        if feat is not None:
            X.append(feat)
            y.append(canonical_class)

X = np.array(X)
y = np.array(y)
print(f'Ekstraksi Selesai! Total sampel: {len(X)} | Dimensi HOG: {X.shape[1]}')


## **3. Split Data (Train & Test)**
Bagi data 80% pelatihan dan 20% pengujian dengan pembagian merata (*stratified*).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Jumlah Data Latih: {len(X_train)}')
print(f'Jumlah Data Uji  : {len(X_test)}')


## **4. Training & Evaluasi: SVM vs Random Forest**
Melatih kedua model sesuai deliverables jurnal dan PRD Notion untuk perbandingan performa.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. MODEL SUPPORT VECTOR MACHINE (SVM)
print('Melatih model SVM (kernel RBF, probability=True)...')
svm_model = SVC(kernel='rbf', C=10.0, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f'Akurasi SVM: {svm_acc * 100:.2f}%')

# 2. MODEL RANDOM FOREST
print('Melatih model Random Forest (100 estimators)...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f'Akurasi Random Forest: {rf_acc * 100:.2f}%')

# 3. TABEL EVALUASI SVM
print('=== CLASSIFICATION REPORT (SVM) ===')
print(classification_report(y_test, svm_preds))

# 4. VISUALISASI CONFUSION MATRIX
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, svm_preds, labels=svm_model.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=svm_model.classes_, yticklabels=svm_model.classes_)
plt.title(f'Confusion Matrix SVM (Akurasi: {svm_acc * 100:.2f}%)')
plt.xlabel('Prediksi Model')
plt.ylabel('Label Sebenarnya')
plt.tight_layout()
plt.show()


## **5. Export Model Artifact (`face_shape_svm.pkl`)**
Simpan model SVM terlatih dan download langsung ke komputermu.

In [ ]:
import joblib
from google.colab import files

MODEL_FILE = 'face_shape_svm.pkl'
joblib.dump(svm_model, MODEL_FILE)
print(f'Model berhasil diekspor sebagai {MODEL_FILE}')

# Download otomatis
files.download(MODEL_FILE)
